# 🎙️ SLAM-ASR Turkish: Common Voice Benchmark Notebook

Bu notebook, **SLAM-LLM** (Speech-Language Model Framework) metodolojisini birebir uygulayarak **Common Voice Türkçe** (Local / Drive Corpus 25.0) veri setinde state-of-the-art (SOTA) ASR başarımını elde etmeniz için tasarlanmıştır.

### ⚡ Özellikler:
- Kodlar doğrudan **GitHub** üzerinden Colab ortamına klonlanır.
- Veri seti Google Drive üzerindeki `datasets/speech/CommonVoice/cv-corpus-25.0-2026-03-09/tr` dizininden okunur.
- Eğitilen model ağırlıkları ve checkpoint'ler doğrudan **Hugging Face Hub** hesabınıza otomatik yüklenir.
- **Speech Encoder:** `openai/whisper-large-v3` (Frozen)
- **Projector:** 5x Frame Concatenation Linear Projector (Trainable)
- **LLM Backbone:** `Qwen/Qwen2.5-7B-Instruct` (LoRA Fine-tuning)

---

## 1. Donanım & Ekran Kartı Kontrolü (GPU Check)

In [ ]:
!nvidia-smi

## 2. Google Drive Bağlama & GitHub'dan Klonlama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/gokhanersoy/slam-asr.git || true
%cd /content/slam-asr/commonvoice_tr

## 3. Gerekli Kütüphanelerin Kurulumu

In [ ]:
!pip install -q -r requirements.txt

## 4. Hugging Face Girişi (Modeli HF Hub'a Yüklemek İçin) 🔑

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        from huggingface_hub import login
        login(token=token)
        print("✅ HF_TOKEN Colab Secrets'tan otomatik yüklendi!")
    else:
        from huggingface_hub import notebook_login
        notebook_login()
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 5. Google Drive Üzerindeki Common Voice Türkçe Veri Setinin Hazırlanması
Google Drive `datasets/speech/CommonVoice/cv-corpus-25.0-2026-03-09/tr` dizinindeki TSV dosyaları ve klipsler okunur, Colab hızlı SSD diskinde `data/` manifestoları oluşturulur.

In [ ]:
# Drive üzerindeki dizin yolu
CV_DRIVE_DIR = "/content/drive/MyDrive/datasets/speech/CommonVoice/cv-corpus-25.0-2026-03-09/tr"

# Veri hazırlama betiğini çalıştırın
!python prepare_data.py --cv_dir {CV_DRIVE_DIR} --output_dir data

## 6. Model Eğitimi (SLAM-ASR Fine-Tuning)
Whisper Large V3 dondurulur, 2 katmanlı Linear Projector katmanı eğitilir ve Qwen2.5-7B-Instruct modeline LoRA uygulanır.

In [ ]:
!python train.py train_config.num_epochs=5 train_config.batch_size_training=4 train_config.gradient_accumulation_steps=4

## 7. Test Setinde Değerlendirme (WER & CER Hesaplama)

In [ ]:
!python evaluate.py --checkpoint_path checkpoints/slam_qwen_tr/best_checkpoint.pt --output_file evaluation_results.json

## 8. Eğitilen Modeli Hugging Face Hub'a Yükleme 🚀
Eğitilen model ağırlıklarını tek komutla kendi Hugging Face hesabınıza yükleyin.

In [ ]:
# Kendi Hugging Face Kullanıcı Adınızı Yazın (Örn: gokhanersoy/slam-asr-tr-commonvoice)
HF_REPO_ID = "kullanici_adiniz/slam-asr-tr-commonvoice"

!python push_to_hub.py --repo_id {HF_REPO_ID} --checkpoint_dir checkpoints/slam_qwen_tr

## 9. Örnek Deşifre Playground (Single Audio Inference)

In [ ]:
import torch
import whisper
from omegaconf import OmegaConf
from src.models.slam_model import model_factory

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = OmegaConf.load("conf/prompt.yaml")

# Modeli yükle
model, tokenizer = model_factory(
    train_config=cfg.train_config,
    model_config=cfg.model_config,
    ckpt_path="checkpoints/slam_qwen_tr/best_checkpoint.pt"
)
model.to(device)
model.eval()

# Örnek Ses Dosyası Yolu
sample_audio_path = "data/audio/test/test_000000.wav"

# Log Mel Spectrogram
audio_raw = whisper.load_audio(sample_audio_path)
audio_raw = whisper.pad_or_trim(audio_raw)
audio_mel = whisper.log_mel_spectrogram(audio_raw, n_mels=128).permute(1, 0).unsqueeze(0).to(device)

# Tahmin Al
prediction = model.generate_transcription(audio_mel=audio_mel)
print(f"🎙️ Tahmin Edilen Metin: {prediction}")
